# Module 1.2: Simulating HMM Sequences

## Learning Objectives
By completing this notebook, you will:
1. Implement forward simulation of HMM sequences
2. Understand the generative process: states → observations
3. Visualize statistical properties of simulated data
4. Analyze state persistence and transition dynamics
5. Compare theoretical and empirical distributions

## Prerequisites
- HMM definition and notation (Module 1.1)
- Basic probability distributions
- Numpy random sampling

## Estimated Time: 50 minutes

---

In [ ]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from typing import Tuple, List

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. The HMM Generative Process

Simulating an HMM follows this algorithm:

**Input**: Model λ = (A, B, π), length T  
**Output**: State sequence Q, observation sequence O

```
1. Sample initial state: q₁ ~ Categorical(π)
2. Sample first observation: o₁ ~ Categorical(B[q₁, :])
3. For t = 2 to T:
     a. Sample state: q_t ~ Categorical(A[q_{t-1}, :])
     b. Sample observation: o_t ~ Categorical(B[q_t, :])
4. Return Q = [q₁, ..., q_T], O = [o₁, ..., o_T]
```

This is **forward simulation** - the natural generative direction.

In [ ]:
# Purpose: Implement HMM forward simulation
# Key Concept: Sample from categorical distributions

class DiscreteHMM:
    """Discrete HMM with simulation capability."""
    
    def __init__(self, states, observations, A, B, pi):
        self.states = states
        self.observations = observations
        self.K = len(states)
        self.M = len(observations)
        self.A = A
        self.B = B
        self.pi = pi
        
        self.state_to_idx = {s: i for i, s in enumerate(states)}
        self.obs_to_idx = {o: i for i, o in enumerate(observations)}
    
    def simulate(self, T: int, return_indices: bool = False, 
                 random_seed: int = None) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate HMM sequence of length T.
        
        Parameters
        ----------
        T : int
            Sequence length
        return_indices : bool
            If True, return integer indices; else return labels
        random_seed : int, optional
            Random seed for reproducibility
        
        Returns
        -------
        state_seq : ndarray (T,)
            Hidden state sequence
        obs_seq : ndarray (T,)
            Observation sequence
        """
        if random_seed is not None:
            np.random.seed(random_seed)
        
        # Allocate arrays
        state_seq = np.zeros(T, dtype=int)
        obs_seq = np.zeros(T, dtype=int)
        
        # Step 1: Sample initial state
        state_seq[0] = np.random.choice(self.K, p=self.pi)
        
        # Step 2: Sample initial observation
        obs_seq[0] = np.random.choice(self.M, p=self.B[state_seq[0], :])
        
        # Step 3: Forward simulation
        for t in range(1, T):
            # Sample next state based on previous state
            state_seq[t] = np.random.choice(self.K, p=self.A[state_seq[t-1], :])
            
            # Sample observation from current state
            obs_seq[t] = np.random.choice(self.M, p=self.B[state_seq[t], :])
        
        # Convert to labels if requested
        if return_indices:
            return state_seq, obs_seq
        else:
            state_labels = np.array([self.states[i] for i in state_seq])
            obs_labels = np.array([self.observations[i] for i in obs_seq])
            return state_labels, obs_labels

# Define weather HMM
states = ['Sunny', 'Rainy']
observations = ['Walk', 'Shop', 'Clean']

A = np.array([
    [0.7, 0.3],
    [0.4, 0.6]
])

B = np.array([
    [0.6, 0.3, 0.1],
    [0.1, 0.4, 0.5]
])

pi = np.array([0.6, 0.4])

weather_hmm = DiscreteHMM(states, observations, A, B, pi)

# Simulate sequence
T = 30
state_seq, obs_seq = weather_hmm.simulate(T, random_seed=42)

print("Simulated HMM Sequence (T=30)")
print("="*70)
print("\nDay  State     Observation")
print("-"*70)
for i in range(T):
    print(f"{i+1:3d}  {state_seq[i]:8s}  {obs_seq[i]}")

## 2. Visualizing Simulated Sequences

Let's create comprehensive visualizations to understand sequence structure.

In [ ]:
# Purpose: Visualize HMM simulation output
# Key Concept: Hidden states generate observable patterns

def visualize_hmm_simulation(hmm, T=100, random_seed=42):
    """
    Create comprehensive visualization of HMM simulation.
    """
    # Simulate
    state_seq, obs_seq = hmm.simulate(T, random_seed=random_seed)
    state_idx, obs_idx = hmm.simulate(T, return_indices=True, random_seed=random_seed)
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    
    # Plot 1: Hidden states
    ax1 = axes[0]
    ax1.step(range(T), state_idx, where='mid', linewidth=2, color='blue', alpha=0.7)
    ax1.fill_between(range(T), state_idx, alpha=0.3, step='mid', color='blue')
    ax1.set_ylabel('Hidden State', fontsize=12)
    ax1.set_title('Hidden State Sequence', fontsize=14, fontweight='bold')
    ax1.set_yticks(range(hmm.K))
    ax1.set_yticklabels(hmm.states)
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, T-1)
    
    # Plot 2: Observations
    ax2 = axes[1]
    colors = plt.cm.Set3(range(hmm.M))
    for i in range(T):
        ax2.bar(i, 1, color=colors[obs_idx[i]], edgecolor='black', linewidth=0.5)
    ax2.set_ylabel('Observation', fontsize=12)
    ax2.set_title('Observation Sequence', fontsize=14, fontweight='bold')
    ax2.set_yticks([])
    ax2.set_xlim(-0.5, T-0.5)
    
    # Create legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=colors[i], edgecolor='black', 
                             label=hmm.observations[i]) 
                       for i in range(hmm.M)]
    ax2.legend(handles=legend_elements, loc='upper right', ncol=hmm.M)
    ax2.grid(True, alpha=0.3, axis='x')
    
    # Plot 3: Aligned view
    ax3 = axes[2]
    ax3.scatter(range(T), state_idx + 0.1, s=100, c='blue', 
               alpha=0.6, label='Hidden State', marker='o')
    ax3.scatter(range(T), obs_idx / hmm.M * hmm.K - 0.1, s=100, 
               c=[colors[i] for i in obs_idx], 
               alpha=0.8, label='Observation', marker='s', edgecolors='black')
    
    for t in range(T):
        ax3.plot([t, t], [state_idx[t] + 0.1, obs_idx[t] / hmm.M * hmm.K - 0.1],
                'gray', alpha=0.2, linewidth=1)
    
    ax3.set_xlabel('Time', fontsize=12)
    ax3.set_ylabel('State / Obs Index', fontsize=12)
    ax3.set_title('State-Observation Alignment', fontsize=14, fontweight='bold')
    ax3.legend(loc='upper right')
    ax3.grid(True, alpha=0.3)
    ax3.set_xlim(0, T-1)
    
    plt.tight_layout()
    plt.show()
    
    return state_seq, obs_seq

# Visualize
states, obs = visualize_hmm_simulation(weather_hmm, T=100, random_seed=123)

## 3. Statistical Properties of Simulations

Compare empirical distributions from simulation to theoretical parameters.

In [ ]:
# Purpose: Analyze statistical properties of simulated sequences
# Key Concept: Law of large numbers - empirical → theoretical

def analyze_simulation_statistics(hmm, T=10000, random_seed=42):
    """
    Compare empirical and theoretical distributions.
    """
    # Simulate long sequence
    state_seq, obs_seq = hmm.simulate(T, return_indices=True, random_seed=random_seed)
    
    print("STATISTICAL ANALYSIS OF HMM SIMULATION")
    print("="*70)
    print(f"Sequence length: T = {T:,}\n")
    
    # 1. State distribution
    print("1. STATE DISTRIBUTION")
    print("-"*70)
    state_counts = Counter(state_seq)
    print(f"{'State':<15s} {'Empirical':>12s} {'Theoretical':>12s} {'Difference':>12s}")
    print("-"*70)
    
    # Compute stationary distribution
    # π_stationary satisfies: π_stationary @ A = π_stationary
    eigenvalues, eigenvectors = np.linalg.eig(hmm.A.T)
    stationary_idx = np.argmax(eigenvalues)
    stationary = np.real(eigenvectors[:, stationary_idx])
    stationary = stationary / stationary.sum()
    
    for i, state in enumerate(hmm.states):
        empirical = state_counts[i] / T
        theoretical = stationary[i]
        diff = empirical - theoretical
        print(f"{state:<15s} {empirical:>12.4f} {theoretical:>12.4f} {diff:>+12.4f}")
    
    # 2. Transition frequencies
    print("\n2. TRANSITION FREQUENCIES")
    print("-"*70)
    transition_counts = np.zeros((hmm.K, hmm.K))
    for t in range(T-1):
        transition_counts[state_seq[t], state_seq[t+1]] += 1
    
    empirical_A = transition_counts / transition_counts.sum(axis=1, keepdims=True)
    
    print("Empirical transition matrix:")
    print(empirical_A)
    print("\nTheoretical transition matrix:")
    print(hmm.A)
    print("\nDifference:")
    print(empirical_A - hmm.A)
    
    # 3. Observation distribution
    print("\n3. OBSERVATION DISTRIBUTION (UNCONDITIONAL)")
    print("-"*70)
    obs_counts = Counter(obs_seq)
    
    # Theoretical: π_stationary @ B
    theoretical_obs = stationary @ hmm.B
    
    print(f"{'Observation':<15s} {'Empirical':>12s} {'Theoretical':>12s} {'Difference':>12s}")
    print("-"*70)
    for i, obs in enumerate(hmm.observations):
        empirical = obs_counts[i] / T
        theoretical = theoretical_obs[i]
        diff = empirical - theoretical
        print(f"{obs:<15s} {empirical:>12.4f} {theoretical:>12.4f} {diff:>+12.4f}")
    
    # 4. Conditional observation distributions
    print("\n4. CONDITIONAL OBSERVATION DISTRIBUTIONS")
    print("-"*70)
    for state_idx, state in enumerate(hmm.states):
        print(f"\nGiven state = {state}:")
        state_mask = state_seq == state_idx
        state_obs = obs_seq[state_mask]
        
        if len(state_obs) > 0:
            obs_given_state = Counter(state_obs)
            print(f"{'Observation':<15s} {'Empirical':>12s} {'Theoretical':>12s}")
            for obs_idx, obs in enumerate(hmm.observations):
                empirical = obs_given_state[obs_idx] / len(state_obs)
                theoretical = hmm.B[state_idx, obs_idx]
                print(f"  {obs:<13s} {empirical:>12.4f} {theoretical:>12.4f}")
    
    print("\n" + "="*70)
    
    return state_seq, obs_seq, empirical_A

# Run analysis
states_long, obs_long, emp_A = analyze_simulation_statistics(weather_hmm, T=10000)

## 4. State Persistence and Run Length Analysis

How long do states typically persist before transitioning?

In [ ]:
# Purpose: Analyze state persistence (dwell times)
# Key Concept: Geometric distribution from Markov chain

def analyze_state_persistence(state_seq, states):
    """
    Compute run length distributions for each state.
    """
    # Find all runs
    runs = {state: [] for state in range(len(states))}
    
    current_state = state_seq[0]
    current_length = 1
    
    for t in range(1, len(state_seq)):
        if state_seq[t] == current_state:
            current_length += 1
        else:
            runs[current_state].append(current_length)
            current_state = state_seq[t]
            current_length = 1
    
    # Add final run
    runs[current_state].append(current_length)
    
    # Theoretical mean: E[dwell time] = 1 / (1 - A_ii)
    print("STATE PERSISTENCE ANALYSIS")
    print("="*70)
    
    for state_idx, state_name in enumerate(states):
        if len(runs[state_idx]) > 0:
            run_lengths = runs[state_idx]
            mean_empirical = np.mean(run_lengths)
            std_empirical = np.std(run_lengths)
            
            # Theoretical (geometric distribution)
            # E[X] = 1/p where p = 1 - A_ii
            p_stay = weather_hmm.A[state_idx, state_idx]
            mean_theoretical = 1 / (1 - p_stay) if p_stay < 1 else np.inf
            
            print(f"\n{state_name}:")
            print(f"  Number of runs: {len(run_lengths)}")
            print(f"  Mean duration (empirical): {mean_empirical:.2f}")
            print(f"  Mean duration (theoretical): {mean_theoretical:.2f}")
            print(f"  Std deviation: {std_empirical:.2f}")
            print(f"  Min/Max: {min(run_lengths)}/{max(run_lengths)}")
    
    print("\n" + "="*70)
    
    # Visualize
    fig, axes = plt.subplots(1, len(states), figsize=(12, 4))
    
    for state_idx, state_name in enumerate(states):
        ax = axes[state_idx] if len(states) > 1 else axes
        
        if len(runs[state_idx]) > 0:
            ax.hist(runs[state_idx], bins=30, alpha=0.7, edgecolor='black')
            ax.axvline(np.mean(runs[state_idx]), color='red', 
                      linestyle='--', linewidth=2, label='Empirical mean')
            
            p_stay = weather_hmm.A[state_idx, state_idx]
            theoretical_mean = 1 / (1 - p_stay)
            ax.axvline(theoretical_mean, color='blue',
                      linestyle='--', linewidth=2, label='Theoretical mean')
            
            ax.set_xlabel('Run Length', fontsize=11)
            ax.set_ylabel('Frequency', fontsize=11)
            ax.set_title(f'{state_name} State Persistence', fontsize=12)
            ax.legend()
            ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return runs

# Analyze
runs = analyze_state_persistence(states_long, states)

## Exercise 1: Multi-Sequence Simulation

**Task:** Simulate multiple independent sequences and compute confidence intervals for the stationary distribution.

Generate N=100 sequences of length T=1000 each, compute state frequencies for each, then report:
1. Mean frequency across sequences
2. 95% confidence interval
3. Compare to theoretical stationary distribution

**Expected Output:** Statistical summary with confidence intervals.

In [ ]:
# YOUR CODE HERE
# ---------------

def multi_sequence_analysis(hmm, N=100, T=1000):
    """
    Simulate N independent sequences and analyze variability.
    
    Parameters
    ----------
    hmm : DiscreteHMM
        The HMM model
    N : int
        Number of sequences
    T : int
        Length of each sequence
    
    Returns
    -------
    frequencies : ndarray (N, K)
        State frequencies for each sequence
    confidence_intervals : ndarray (K, 2)
        95% CI for each state
    """
    # YOUR IMPLEMENTATION HERE
    # For each of N sequences:
    #   - Simulate
    #   - Count state frequencies
    # Compute mean and 95% CI (use np.percentile)
    
    return None, None

frequencies, cis = multi_sequence_analysis(weather_hmm, N=100, T=1000)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_1():
    assert frequencies is not None, "Don't forget to compute frequencies!"
    assert cis is not None, "Don't forget to compute confidence intervals!"
    
    assert frequencies.shape == (100, 2), "Should have 100 sequences x 2 states"
    assert cis.shape == (2, 2), "Should have 2 states x 2 bounds"
    
    # Check that frequencies sum to 1
    assert np.allclose(frequencies.sum(axis=1), 1), "Frequencies should sum to 1"
    
    # Check CI ordering
    assert np.all(cis[:, 0] <= cis[:, 1]), "Lower bound should be <= upper bound"
    
    print("✅ Exercise 1 passed!")
    print("\nMulti-Sequence Analysis:")
    print("="*60)
    
    # Compute stationary distribution
    eigenvalues, eigenvectors = np.linalg.eig(weather_hmm.A.T)
    stationary_idx = np.argmax(eigenvalues)
    stationary = np.real(eigenvectors[:, stationary_idx])
    stationary = stationary / stationary.sum()
    
    for i, state in enumerate(states):
        mean_freq = frequencies[:, i].mean()
        ci_low, ci_high = cis[i]
        theoretical = stationary[i]
        
        print(f"\n{state}:")
        print(f"  Mean frequency: {mean_freq:.4f}")
        print(f"  95% CI: [{ci_low:.4f}, {ci_high:.4f}]")
        print(f"  Theoretical: {theoretical:.4f}")
        print(f"  In CI: {'Yes' if ci_low <= theoretical <= ci_high else 'No'}")

test_exercise_1()

## Exercise 2: Financial Market Simulation

**Task:** Create and simulate a 3-state market regime HMM:
- States: {Bull, Bear, Sideways}
- Observations: Discretized returns {Large+, Small+, Neutral, Small-, Large-}

Simulate T=252 days (1 trading year) and visualize:
1. State sequence
2. Observation sequence  
3. State durations

**Expected Output:** Complete market simulation with analysis.

In [ ]:
# YOUR CODE HERE
# ---------------

# Define market HMM
market_states = ['Bull', 'Bear', 'Sideways']
market_obs = ['Large+', 'Small+', 'Neutral', 'Small-', 'Large-']

# YOUR MATRICES HERE
A_market = None  # 3x3 transition matrix
B_market = None  # 3x5 emission matrix
pi_market = None # 3-element initial distribution

# Create and simulate
if A_market is not None:
    market_hmm = DiscreteHMM(market_states, market_obs, A_market, B_market, pi_market)
    market_state_seq, market_obs_seq = market_hmm.simulate(252, random_seed=42)
else:
    market_hmm = None
    market_state_seq = None
    market_obs_seq = None

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_2():
    assert market_hmm is not None, "Don't forget to create market HMM!"
    assert market_state_seq is not None, "Don't forget to simulate!"
    assert market_obs_seq is not None, "Don't forget observations!"
    
    assert market_hmm.K == 3, "Should have 3 states"
    assert market_hmm.M == 5, "Should have 5 observations"
    assert len(market_state_seq) == 252, "Should simulate 252 days"
    
    # Verify distributions
    assert np.allclose(market_hmm.A.sum(axis=1), 1), "A rows should sum to 1"
    assert np.allclose(market_hmm.B.sum(axis=1), 1), "B rows should sum to 1"
    assert np.isclose(market_hmm.pi.sum(), 1), "pi should sum to 1"
    
    print("✅ Exercise 2 passed!")
    print("\nMarket Simulation Summary:")
    print("="*60)
    
    # Convert to indices for counting
    state_idx = [market_hmm.state_to_idx[s] for s in market_state_seq]
    state_counts = Counter(state_idx)
    
    for idx, state in enumerate(market_states):
        count = state_counts[idx]
        pct = 100 * count / 252
        print(f"  {state:10s}: {count:3d} days ({pct:5.1f}%)")
    
    # Visualize
    visualize_hmm_simulation(market_hmm, T=252, random_seed=42)

test_exercise_2()

## Exercise 3: Convergence to Stationarity

**Task:** Demonstrate convergence to stationary distribution:
1. Simulate very long sequence (T=100,000)
2. Compute cumulative state frequencies at multiple time points
3. Plot convergence to stationary distribution
4. Estimate convergence rate

**Expected Output:** Convergence plot and rate estimate.

In [ ]:
# YOUR CODE HERE
# ---------------

def analyze_convergence(hmm, T=100000, checkpoints=50):
    """
    Analyze convergence to stationary distribution.
    
    Parameters
    ----------
    hmm : DiscreteHMM
        The HMM model
    T : int
        Total sequence length
    checkpoints : int
        Number of checkpoints to evaluate
    
    Returns
    -------
    checkpoint_times : ndarray
        Time points evaluated
    cumulative_freqs : ndarray (checkpoints, K)
        Cumulative frequencies at each checkpoint
    """
    # YOUR IMPLEMENTATION HERE
    # Simulate long sequence
    # At logarithmically-spaced checkpoints, compute cumulative frequencies
    # Return checkpoint times and frequencies
    
    return None, None

checkpoint_times, cumulative_freqs = analyze_convergence(weather_hmm)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_3():
    assert checkpoint_times is not None, "Don't forget checkpoint times!"
    assert cumulative_freqs is not None, "Don't forget frequencies!"
    
    assert len(checkpoint_times) == 50, "Should have 50 checkpoints"
    assert cumulative_freqs.shape == (50, 2), "Should be 50 x 2"
    
    # Check that later frequencies are closer to stationary
    eigenvalues, eigenvectors = np.linalg.eig(weather_hmm.A.T)
    stationary_idx = np.argmax(eigenvalues)
    stationary = np.real(eigenvectors[:, stationary_idx])
    stationary = stationary / stationary.sum()
    
    early_error = np.linalg.norm(cumulative_freqs[0] - stationary)
    late_error = np.linalg.norm(cumulative_freqs[-1] - stationary)
    
    assert late_error < early_error, "Should converge toward stationary"
    
    print("✅ Exercise 3 passed!")
    print(f"\nConvergence analysis:")
    print(f"  Initial error: {early_error:.6f}")
    print(f"  Final error: {late_error:.6f}")
    print(f"  Improvement: {100*(1 - late_error/early_error):.1f}%")
    
    # Plot convergence
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for state_idx, state in enumerate(states):
        ax.semilogx(checkpoint_times, cumulative_freqs[:, state_idx],
                    marker='o', label=f'{state} (empirical)', alpha=0.7)
        ax.axhline(stationary[state_idx], linestyle='--', 
                   label=f'{state} (stationary)', alpha=0.7)
    
    ax.set_xlabel('Time', fontsize=12)
    ax.set_ylabel('Cumulative Frequency', fontsize=12)
    ax.set_title('Convergence to Stationary Distribution', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

test_exercise_3()

## Summary

### Key Takeaways

1. **Forward simulation** generates sequences by sampling from conditional distributions
2. **Law of large numbers**: Empirical statistics → theoretical parameters as T → ∞
3. **State persistence** follows geometric distribution determined by A_ii
4. **Stationary distribution**: Long-run state frequencies (left eigenvector of A)
5. **Convergence**: Initial distribution π washes out over time

### Simulation Recipe

```python
# Initialize
q_1 ~ Categorical(π)
o_1 ~ Categorical(B[q_1, :])

# Iterate
for t in range(2, T+1):
    q_t ~ Categorical(A[q_{t-1}, :])
    o_t ~ Categorical(B[q_t, :])
```

### Statistical Properties

- **Expected run length**: E[dwell in state i] = 1 / (1 - A_ii)
- **Stationary distribution**: π_∞ such that π_∞ᵀ A = π_∞ᵀ
- **Mixing time**: Time to reach stationarity depends on second eigenvalue of A

### Applications

Simulation is used for:
1. **Testing algorithms**: Verify Forward-Backward, Viterbi on known sequences
2. **Monte Carlo methods**: Estimate quantities via simulation
3. **Stress testing**: Generate synthetic scenarios
4. **Bootstrap**: Resample from fitted model

### Next Steps

Now that we can generate data, Module 2 covers the inverse problems:
- **Forward algorithm**: Compute P(O | λ) from observations
- **Viterbi algorithm**: Infer most likely state sequence
- **Baum-Welch**: Learn parameters from observations

---